## 7.2 Evaluate Test Dataset & Calculate Accuracy

Sau khi huấn luyện, chúng ta cần đánh giá toàn diện mô hình trên tập dữ liệu kiểm thử (Test Dataset) để đo lường khả năng tổng quát hóa của nó. Độ chính xác (Accuracy) được tính theo công thức:

$$ \text{Accuracy} = \frac{\text{Correct predictions}}{\text{Total predictions}} \times 100\% $$

In [10]:
import torch

# Chuyển mô hình sang chế độ đánh giá (evaluation mode)
model.eval()

correct_predictions = 0
total_predictions = 0

# Khởi tạo list lưu trữ tất cả nhãn thực tế và dự đoán cho phần Confusion Matrix
all_labels = []
all_preds = []

# Tắt tính toán gradient để tiết kiệm bộ nhớ và tăng tốc độ
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        
        # Đưa dữ liệu qua mô hình
        outputs = model(images)
        
        # Lấy class có xác suất cao nhất
        _, predicted = torch.max(outputs, 1)
        
        # Cập nhật số lượng
        total_predictions += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()
        
        # Lưu lại để dùng cho Error Analysis
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(predicted.cpu().numpy())

# Tính và in kết quả theo format yêu cầu
test_accuracy = (correct_predictions / total_predictions) * 100
print(f"Test Accuracy: {test_accuracy:.2f}%")

NameError: name 'test_loader' is not defined

## 7.3 Make Predictions (Predicted vs Actual)

Để có cái nhìn trực quan, chúng ta sẽ trích xuất 16 hình ảnh ngẫu nhiên từ tập Test và so sánh nhãn dự đoán (Predicted) với nhãn thực tế (Actual). 
*   **Correct ✓**: Dự đoán đúng (Màu xanh)
*   **Wrong ✗**: Dự đoán sai (Màu đỏ)

In [11]:
import matplotlib.pyplot as plt
import numpy as np

# Danh sách các class của FashionMNIST
classes = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat", 
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

# Lấy ra 1 batch dữ liệu test để trực quan hóa
dataiter = iter(test_loader)
images, labels = next(dataiter)

# Lấy 16 ảnh đầu tiên
images_to_show = images[:16]
labels_to_show = labels[:16]

# Dự đoán 16 ảnh này
model.eval()
with torch.no_grad():
    outputs = model(images_to_show.to(device))
    _, preds = torch.max(outputs, 1)

# Vẽ biểu đồ 4x4
fig = plt.figure(figsize=(10, 12))
for idx in np.arange(16):
    ax = fig.add_subplot(4, 4, idx+1, xticks=[], yticks=[])
    
    # Xử lý tensor ảnh về numpy array để hiển thị
    img = images_to_show[idx].squeeze().numpy()
    ax.imshow(img, cmap='gray')
    
    actual_label = classes[labels_to_show[idx]]
    pred_label = classes[preds[idx]]
    
    # Format hiển thị đúng/sai
    if actual_label == pred_label:
        status = "✓"
        color = "green"
    else:
        status = "✗"
        color = "red"
        
    # Tiêu đề cho từng ảnh phụ
    title_text = f"Actual: {actual_label}\nPredicted: {pred_label}\n{status}"
    ax.set_title(title_text, color=color, fontsize=11)

plt.tight_layout()
plt.show()

NameError: name 'test_loader' is not defined

## 7.4 Error Analysis: Confusion Matrix (Mở rộng)

Ma trận nhầm lẫn (Confusion Matrix) giúp chúng ta phân tích lỗi chi tiết hơn, xác định rõ ràng mô hình thường hay nhầm lẫn giữa các cặp class nào nhất.

In [12]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

# Sử dụng all_labels và all_preds từ Cell 20
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", 
            xticklabels=classes, yticklabels=classes)
plt.title("Confusion Matrix - FashionMNIST Test Dataset")
plt.ylabel("Actual Label")
plt.xlabel("Predicted Label")
plt.tight_layout()
plt.show()

ValueError: Found empty input array (e.g., `y_true` or `y_pred`) while a minimum of 1 sample is required.